In [109]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf senetnce-transformers chromadb

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.14.3 (v3.14.3:323c59a5e34, Feb  3 2026, 11:41:37) [Clang 16.0.0 (clang-1600.0.26.6)]
/Users/sagarbansal/rag-env/bin/python


In [2]:
import os
import uuid
import chromadb

In [3]:
import sys

print(sys.version)
print(sys.executable)


3.14.3 (v3.14.3:323c59a5e34, Feb  3 2026, 11:41:37) [Clang 16.0.0 (clang-1600.0.26.6)]
/Users/sagarbansal/rag-env/bin/python


In [4]:
from langchain_core.documents import Document

In [5]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {"source": "https://www.google.com"}
)

In [6]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [7]:
type(sample_doc)

langchain_core.documents.base.Document

In [8]:
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("data/python.txt", encoding="utf-8")

/var/folders/vj/dwnt486d1651_5fw17gxqwz80000gn/T/ipykernel_55508/2919498118.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
/Users/sagarbansal/rag-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
document = loader.load()

In [10]:
from langchain_community.document_loaders.pdf import PyPDFLoader
pdf_loader = PyPDFLoader("data/research2.pdf")

document = pdf_loader.load()

# Ingestion Pipeline

# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [11]:
def load_all_pdfs():
     folder_path = "data/pdfs"
     num_docs = 0
     all_docs = []

     for filename in os.listdir(folder_path):
         if filename.lower().endswith(".pdf"):

           pdf_path = os.path.join(folder_path,filename)

           loader = PyPDFLoader(pdf_path)
           doc = loader.load()
        
           all_docs.extend(doc)
           num_docs += 1
         
     print("total pdfs:", num_docs)
     print("total pages:", len(all_docs))
     return all_docs


In [12]:
all_pdf_documents = load_all_pdfs()

total pdfs: 1
total pages: 23


### Chunking

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [14]:
chunks = split_doc(all_pdf_documents)
len(chunks)

82

### Embedding

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
class EmbeddingManager:
    def __init__(self, model_name ="all-MiniLM-L6-v2"):
         self.model_name = model_name
         print("loading model....", self.model_name)
         self.model = SentenceTransformer(self.model_name) 
         print("embedding dimensions", self.model.get_embedding_dimension())
        
    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [17]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 11971.44it/s]


embedding dimensions 384


### Vector Store

In [18]:
import os
import uuid
import chromadb

In [19]:
import os
import uuid
import chromadb


class VectorStoreManager:

    def __init__(
        self,
        persist_directory="data/vector_store",
        collection_name="pdf_documents"
    ):
        # Store collection name
        self.collection_name = collection_name

        # Store database directory
        self.persist_directory = persist_directory

        # Initially no client
        self.client = None

        # Initially no collection
        self.collection = None

        # Initialize ChromaDB
        self.initialize_store()


    def initialize_store(self):

        # Create directory if it doesn't exist
        os.makedirs(
            self.persist_directory,
            exist_ok=True
        )

        # Create persistent ChromaDB client
        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        # Create or get collection
        # Using COSINE distance metric
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "Vector store collection for PDF embeddings in RAG",

                # Distance metric
                "hnsw:space": "cosine"
            }
        )

        print(
            "Initialized the vector store with collection:",
            self.collection_name
        )

        print(
            "Distance metric: cosine"
        )

        print(
            "Docs in collection:",
            self.collection.count()
        )


    def add_documents(self, documents, embeddings):

        # Check that every document has an embedding
        if len(documents) != len(embeddings):

            raise ValueError(
                "Number of documents doesn't match with number of embeddings"
            )


        # Lists required for ChromaDB
        ids = []

        all_metadata = []

        documents_content = []

        embeddings_list = []


        # Process every document and its corresponding embedding
        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Create a unique document ID
            doc_id = f"doc_{uuid.uuid4()}"

            ids.append(doc_id)


            # Copy existing metadata
            metadata = dict(doc.metadata)


            # Add additional metadata
            metadata["doc_index"] = i

            metadata["content_length"] = len(
                doc.page_content
            )

            all_metadata.append(metadata)


            # Store actual document/chunk text
            documents_content.append(
                doc.page_content
            )


            # Convert NumPy embedding to Python list
            embeddings_list.append(
                embedding.tolist()
            )


        # Add all documents to ChromaDB
        self.collection.add(

            ids=ids,

            metadatas=all_metadata,

            documents=documents_content,

            embeddings=embeddings_list
        )


        print(
            "Total documents added in vector store:",
            len(documents_content)
        )


        print(
            "Total docs in collection:",
            self.collection.count()
        )

In [20]:
vector_store = VectorStoreManager()

Initialized the vector store with collection: pdf_documents
Distance metric: cosine
Docs in collection: 82


In [21]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00,  6.12it/s]

embeddings shape: (82, 384)
Total documents added in vector store: 82
Total docs in collection: 164


### RETRIEVAL PIPELINE

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
class RAGRetriever:

    def __init__(self, emb_manager, vector_store):
        self.emb_manager = emb_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):

        # 1. Convert query into embedding
        query_embedding = self.emb_manager.generate_embeddings(
            [query]
        )[0]


        # 2. Perform semantic search in ChromaDB
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )


        # Store retrieved documents
        retrieved_docs = []


        # 3. Check whether documents were found
        if results["documents"] and results["documents"][0]:

            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]


            # 4. Process every retrieved document
            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):

                # Calculate similarity score
                similarity_score = 1 - distance


                # Apply similarity threshold
                if similarity_score >= score_threshold:

                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })


            print("Retrieved docs:", len(retrieved_docs))


        else:
            print("No documents found")


        return retrieved_docs

In [24]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [25]:
rag_retriever.retrieve("what is RAG?")


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 15.15it/s]

embeddings shape: (1, 384)
Retrieved docs: 5


[{'id': 'doc_d614bd90-eac0-48df-af3c-575a2c7b23ba',
  'document': 'https://docs.langchain.com/oss/python/deepagents/rag 5/23',
  'metadata': {'title': 'Retrieval Augmented Generation (RAG) with Deep Agents - Docs by LangChain',
   'producer': 'Skia/PDF m152',
   'doc_index': 18,
   'page': 4,
   'page_label': '5',
   'source': 'data/pdfs/research.pdf',
   'content_length': 57,
   'total_pages': 23,
   'creationdate': '2026-09-09T09:18:06+00:00',
   'moddate': '2026-09-09T09:18:06+00:00',
   'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36'},
  'distance': 0.6033592224121094,
  'similarity_score': 0.3966407775878906,
  'rank': 1},
 {'id': 'doc_c6a8f703-09ee-4475-ac77-5b6543a86026',
  'document': 'https://docs.langchain.com/oss/python/deepagents/rag 5/23',
  'metadata': {'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36',
   'tit

### INTEGRATE WITH LLMS

#OpenAI-GPT

In [26]:
API_KEY_OPENAI = "<REDACTED>"

In [46]:
# !pip install langchain openai


In [29]:
import sys

print(sys.executable)
print(sys.version)

/Users/sagarbansal/rag-env/bin/python
3.14.3 (v3.14.3:323c59a5e34, Feb  3 2026, 11:41:37) [Clang 16.0.0 (clang-1600.0.26.6)]


In [44]:
# pip install langchain-openai

In [42]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024 
)

In [52]:
## generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    prompt = f""" use given context to generate the answer for the query
                Context : {context}
                Query : {query} """
    resposne = llm.invoke(prompt)
    return response.document

In [53]:
answer = generate_output("what is RAG", rag_retriever,llm)

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 15.51it/s]


embeddings shape: (1, 384)
Retrieved docs: 3


OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [55]:
API_KEY_GROQ = "<REDACTED>"

In [65]:
 # pip install langchain-groq

In [94]:
from langchain_groq import ChatGroq

llm_2 = ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model="qwen/qwen3.8-27b",
    temperature=0.1,
    max_tokens=1024
)

In [95]:
def generate_output(query, rag_retriever, llm, top_k=3):

    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join(
        doc["document"]
        for doc in results
    ) if results else ""

    if not context:
        print("We found no relevant context for the given query")
        return None

    # Create augmented prompt
    prompt = f""" Use the given context to answer the query.
                 Context: {context}
                 Query: {query}
                 Answer:   """

    response = llm.invoke(prompt)

    return response.content

In [96]:
answer = generate_output("what is RAG", rag_retriever,llm_2)

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 20.21it/s]


embeddings shape: (1, 384)
Retrieved docs: 3


In [97]:
print(answer)

Based on the provided context, **RAG** stands for **Retrieval Augmented Generation**.

In the context of Deep Agents, RAG involves orchestrating retrieval, analysis, and synthesis. It allows you to combine primitives such as custom retrieval tools, filesystem backends, subagents, skills, and grading rubrics. The specific approach depends on factors like corpus size, latency requirements, and how strictly answers must be grounded in source data. A common pattern involves indexing a corpus, retrieving relevant chunks at query time, offloading them to the filesystem, and delegating analysis to subagents to keep the orchestrator context clean.


### OLLAMA

In [98]:
pip install langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-ollama]
Note: you may need to restart the kernel to use updated packages.


In [100]:
from langchain_ollama import ChatOllama

In [102]:

llm_3 = ChatOllama(
    model="llama3.2",
    temperature=0.1,
    max_tokens=1024
)

In [107]:
## generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    prompt = f""" use given context to generate the answer for the query
                Context : {context}
                Query : {query} """
    response = llm.invoke(prompt)
    return response.content

In [108]:
answer = generate_output("what is RAG", rag_retriever,llm_3)

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 18.37it/s]


embeddings shape: (1, 384)
Retrieved docs: 3


In [109]:
print(answer)

RAG stands for Retrieval Augmented Generation, which is a framework for building agents that can perform tasks such as answering questions, generating text, and more. It provides primitives for RAG, including custom retrieval tools, a filesystem backend, subagents, and skills grading rubrics.
